<a href="https://colab.research.google.com/github/MBreadd/MBreadd/blob/main/scraping_y_limp_admision_unmsm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping y Construcción del Dataset de Admisión UNMSM

Este notebook forma parte de un proyecto de análisis de los resultados de los procesos de admisión de la Universidad Nacional Mayor de San Marcos (UNMSM).

El objetivo de esta etapa es construir un dataset histórico confiable a partir de los resultados publicados en el portal oficial de admisión. Para ello se desarrolla un pipeline de scraping, limpieza y normalización de datos que permita posteriormente realizar análisis exploratorios y estudios sobre la competitividad de las carreras.

El dataset resultante incluye información de múltiples procesos de admisión entre los años 2023 y 2026.

## 1. Fuente de datos

Los datos utilizados en este proyecto provienen del portal oficial de admisión de la Universidad Nacional Mayor de San Marcos.

Base del sitio:

https://admision.unmsm.edu.pe/

Cada proceso de admisión publica los resultados en páginas específicas que contienen tablas con la lista completa de postulantes.

Estas tablas incluyen información como:

- nombres del postulante
- carrera a la que postuló
- puntaje obtenido en el examen
- orden de mérito
- estado del postulante (ingresó, no ingresó o ausente)

El scraping consiste en extraer automáticamente estas tablas para construir un dataset consolidado que permita analizar varios procesos de admisión de forma conjunta.

## 2. Objetivo del scraping

El objetivo principal del scraping es construir un dataset histórico de postulantes a la UNMSM que permita responder preguntas como:

- ¿Qué carreras son más competitivas?
- ¿Cómo han evolucionado los puntajes de ingreso a lo largo del tiempo?
- ¿Cuál es el puntaje mínimo de ingreso por carrera?
- ¿Cómo se distribuyen los puntajes de los postulantes?

Para lograrlo se extraen los datos de múltiples procesos de admisión y se normalizan para que puedan analizarse de forma conjunta.

## 3. Estructura de los datos extraídos

Cada fila del dataset representa a un postulante inscrito en un proceso de admisión.

Las principales variables son:

- **apellidos**: apellidos del postulante.
- **nombres**: nombres del postulante.
- **carrera**: carrera a la que postuló.
- **puntaje**: puntaje obtenido en el examen de admisión.
- **merito**: orden de mérito dentro de la carrera.
- **observacion**: estado final del postulante.

### Significado de la variable "observacion"

Esta variable indica el resultado del postulante en el proceso de admisión.

Valores posibles:

- **INGRESO** → el postulante obtuvo una vacante.
- **NO ALCANZO VACANTE** → el postulante rindió el examen pero no obtuvo vacante.
- **AUSENTE** → el postulante se inscribió pero no se presentó al examen o no llegó a tiempo.

Es importante notar que el dataset incluye **todos los postulantes inscritos**, independientemente de si rindieron el examen o si ingresaron.

## 4. Pipeline de extracción de datos

El proceso de construcción del dataset sigue el siguiente pipeline:

1. **Definición de los procesos de admisión a analizar**

   Se identifican los enlaces correspondientes a los procesos de admisión disponibles en el portal oficial.

2. **Extracción de las tablas HTML**

   Utilizando `requests` y `BeautifulSoup` se descargan las páginas y se identifican las tablas que contienen los resultados.

3. **Construcción de DataFrames**

   Cada tabla extraída se convierte en un DataFrame de pandas para facilitar su manipulación.

4. **Unificación de los procesos**

   Los DataFrames de cada proceso de admisión se concatenan en un solo dataset.

5. **Limpieza y normalización de datos**

   Se aplican transformaciones para estandarizar nombres de columnas, limpiar texto y convertir variables a sus tipos correctos.

6. **Exportación del dataset final**

   El dataset limpio se guarda en formato CSV para su uso en la etapa de análisis.

## 5. Limpieza y normalización de datos

Una vez extraídos los datos, se realiza un proceso de limpieza que incluye:

- normalización de nombres de columnas
- eliminación de espacios y caracteres inconsistentes
- conversión de puntajes a valores numéricos
- estandarización de nombres de carreras
- manejo de valores faltantes
- eliminación de registros duplicados

Este proceso permite garantizar que el dataset final sea consistente y apto para análisis estadísticos.

## 6. Consideraciones especiales del scraping

### Caso del proceso 2025-2 (Medicina)

En el proceso de admisión **2025-II**, los resultados de la carrera de Medicina fueron publicados en una página distinta al resto de carreras.

Por esta razón se definieron dos fuentes de datos separadas:

- `2025_2` → resultados de todas las carreras excepto Medicina.
- `2025_2M` → resultados de la carrera de Medicina.

Ambos datasets se combinan posteriormente durante la construcción del dataset final.

---

### Cambio en la estructura del portal a partir de 2026

A partir del proceso de admisión **2026-I**, el portal de resultados cambió su estructura web.

Los procesos anteriores publicaban los resultados en tablas HTML estáticas, lo que permitía extraer los datos utilizando herramientas como `requests` y `BeautifulSoup`.

Sin embargo, en el nuevo portal los datos parecen cargarse dinámicamente mediante JavaScript, lo que impide que el método de scraping utilizado en este proyecto pueda extraer la información directamente.

Para extraer estos datos usé la extencion "Web Scraper - Free Web Scraping" de la Chrome web store.

## 7. Dataset final generado

El resultado de este proceso es un dataset consolidado que contiene todos los postulantes de los procesos de admisión analizados.

Este dataset será utilizado en la siguiente etapa del proyecto para realizar:

- análisis exploratorio de datos
- visualización de tendencias
- estudio de la competitividad de las carreras
- análisis de distribución de puntajes

El archivo final se exporta en formato CSV para facilitar su uso en herramientas de análisis y visualización.

# 1. Importaciones y configuración base


In [1]:
!pip install unidecode

In [2]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import html
import os
from io import StringIO
import chardet
import unicodedata
import unidecode

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def configurar_rutas(periodo):
    carpeta_guardado = f"/content/drive/MyDrive/proyecto_admision_unmsm/data/{periodo}"
    os.makedirs(carpeta_guardado, exist_ok=True)

    rutas = {
        "carpeta": carpeta_guardado,
        "carreras": os.path.join(carpeta_guardado, f"carreras_{periodo}.csv"),
        "raw": os.path.join(carpeta_guardado, f"resultados_raw_{periodo}.csv"),
        "limpio": os.path.join(carpeta_guardado, f"resultados_limpio_{periodo}.csv")
    }
    return rutas

URLS_UNMSM = {
    "2023_2": "https://admision.unmsm.edu.pe/WebsiteExa_20232/A.html",
    "2024_1": "https://admision.unmsm.edu.pe/Website20241/A.html",
    "2024_2": "https://admision.unmsm.edu.pe/Website20242/A.html",
    "2025_1": "https://admision.unmsm.edu.pe/Website20251/A.html",
    "2025_2": "https://admision.unmsm.edu.pe/Website20252General/A.html",
    "2025_2M": "https://admision.unmsm.edu.pe/Website20252GeneralA/A.html"
}

# 2. Función para obtener el BeautifulSoup con encoding correcto

In [5]:
# Para get soup
def get_soup(url, headers=None, verbose=True):
    r = requests.get(url, headers=headers)
    enc_auto = chardet.detect(r.content)['encoding']
    r.encoding = enc_auto or 'ISO-8859-1'
    if verbose:
        print(f"{url} -> usando encoding: {r.encoding}")
    return BeautifulSoup(r.text, 'html.parser')

## 3. Scraping de links de carreras


In [6]:
def scraping_carreras(periodo, rutas, headers):
    url_base = URLS_UNMSM.get(periodo, None)
    if not url_base:
        raise ValueError(f"No se encontró URL para {periodo}")

    if periodo == "2025_2M":
        periodo = "2025_2"

    root = "/".join(url_base.split("/")[:-1]) + "/"
    print(f"\nUsando URL base: {url_base}")

    r = requests.get(url_base, headers=headers, timeout=10)
    if r.status_code != 200:
        raise ConnectionError(f"Error HTTP {r.status_code} en {url_base}")

    soup = BeautifulSoup(r.text, "html.parser")
    enlaces = soup.select("table a")

    data = []
    for enlace in enlaces:
        href = enlace.get("href")
        nombre = enlace.text.strip()
        if href and "A/" in href:
            codigo = href.split("/")[-2]
            data.append({
                "codigo": codigo,
                "nombre_carrera": nombre,
                "url": root + href
            })

    if not data:
        raise ValueError(f"No se encontraron carreras en {periodo}")

    df_carreras = pd.DataFrame(data)
    df_carreras.to_csv(rutas["carreras"], index=False, encoding="utf-8-sig")
    print(f"Carreras guardadas: {len(df_carreras)} → {rutas['carreras']}")
    return df_carreras

# 4. Scraping de resultados RAW

In [7]:
# Usando la tabla de carreras
def scraping_resultados(df_carreras, rutas, headers):
    todos_los_resultados = []

    for _, fila in df_carreras.iterrows():
        url = fila["url"]
        try:
            soup = get_soup(url, headers, verbose=False)
            tabla = soup.find("table")
            if tabla:
                df = pd.read_html(StringIO(str(tabla)))[0]
                todos_los_resultados.append(df)
            else:
                print(f"[!] No se encontró tabla en: {url}")
        except Exception as e:
            print(f"[X] Error al procesar {url}: {e}")

    if len(todos_los_resultados) == 0:
        raise RuntimeError("No se extrajo ninguna tabla.")

    df_resultados_raw = pd.concat(todos_los_resultados, ignore_index=True)
    df_resultados_raw.columns = [html.unescape(col).strip() for col in df_resultados_raw.columns]
    df_resultados_raw.to_csv(rutas["raw"], index=False, encoding="utf-8-sig")
    print(f"Resultados RAW guardados en: {rutas['raw']}")
    df_resultados_raw.head()
    return df_resultados_raw

# 5. Limpieza de datos




In [8]:
# Limpiamos los datos
def limpiar_datos(df_raw, rutas):
    df = df_raw.copy()

    df.columns = (
        df.columns.str.lower()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.replace(' ', '', regex=False)
        .str.replace('.', '', regex=False)
    )

    mapping_keywords = {
        'nombre': ['apellidosynombres'],
        'carrera_1': ['escuelaprofesional', 'escuela', 'escuelaprofesionalprimeraopcion', 'escuelaprofesional(primeraopcion)'],
        'carrera_2': ['escuelasegundaopcion', 'escuelaprofesional(segundaopcion)'],
        'puntaje': ['puntajefinal', 'nota'],
        'merito': ['ordendemerito', 'meritoep', 'meritoepalcanzavacante'],
        'obs': ['observacion']
    }

    def map_col_name(col):
        for target, keywords in mapping_keywords.items():
            for kw in keywords:
                if kw == col:
                    return target
        return col

    df = df.rename(columns=lambda c: map_col_name(c))
    df = df.dropna(how="all")

    # Limpiar puntaje
    if "puntaje" in df.columns:
        df["puntaje"] = (
            df["puntaje"].astype(str)
            .str.replace(r"[^\d,.-]", "", regex=True)
            .str.replace(",", ".", regex=False)
        )
        df["puntaje"] = pd.to_numeric(df["puntaje"], errors="coerce")

    # Limpieza general de texto
    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .str.lower()
            .apply(unidecode.unidecode)
        )

    if "obs" in df.columns:
        df["obs"] = df["obs"].replace({
            'alcanzo vacante': 'ingreso',
            'alcanzo vacante primera opcion': 'ingreso',
            'alcanzo vacante segunda opcion': 'ingreso_2',
            'nan': 'no_ingreso'
        })
    else:
        print("Columna 'obs' no encontrada para reemplazo.")

    # Agregar columna 'proceso'
    proceso_actual = os.path.basename(os.path.dirname(rutas["limpio"]))
    if proceso_actual == '2025_2M':
        proceso_actual = '2025_2'
    df["proceso"] = proceso_actual

    df = df.drop_duplicates()
    df = df.drop(columns=["codigo"])

    # Guardar resultado
    df.to_csv(rutas["limpio"], index=False, encoding="utf-8-sig")
    print(f"Resultados LIMPIOS guardados en: {rutas['limpio']}")
    return df


# 6. Revisión / Diagnóstico de los datos (mirar el raw antes de limpiar)

In [9]:
def main(periodo):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept-Language": "es-PE,es;q=0.9,en;q=0.8"
    }

    rutas = configurar_rutas(periodo)

    print("\n" + "━" * 45)
    print(f"-- PROCESO: {periodo}")
    print("━" * 45)

    df_carreras = scraping_carreras(periodo, rutas, headers)
    df_raw = scraping_resultados(df_carreras, rutas, headers)
    display(df_raw.head(3))

    df_final = limpiar_datos(df_raw, rutas)
    display(df_final.head(3))

    print(f"-- Finalizado proceso {periodo} ({df_final.shape[0]} filas limpias)\n")
    return df_final

# 7. EJECUCION

In [10]:
exams = ["2023_2", "2024_1", "2024_2","2025_1","2025_2","2025_2M"]

for exam in exams:
  main(exam)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
-- PROCESO: 2023_2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Usando URL base: https://admision.unmsm.edu.pe/WebsiteExa_20232/A.html
Carreras guardadas: 78 → /content/drive/MyDrive/proyecto_admision_unmsm/data/2023_2/carreras_2023_2.csv
Resultados RAW guardados en: /content/drive/MyDrive/proyecto_admision_unmsm/data/2023_2/resultados_raw_2023_2.csv


,CODIGO,APELLIDOS Y NOMBRES,ESCUELA PROFESIONAL,PUNTAJE FINAL,MERITO E.P,OBSERVACIÓN
0,855168,"ABAD GRANDA, ANDRE FAHET",MEDICINA HUMANA,774.75,NaN,NaN
1,860373,"ABAD NEYRA, ANDREA TAIS",MEDICINA HUMANA,299.00,NaN,NaN
2,894207,"ABAD ONCOY, MARIA BELÉN",MEDICINA HUMANA,441.75,NaN,NaN


Resultados LIMPIOS guardados en: /content/drive/MyDrive/proyecto_admision_unmsm/data/2023_2/resultados_limpio_2023_2.csv


,nombre,carrera_1,puntaje,merito,obs,proceso
0,"abad granda, andre fahet",medicina humana,774.75,NaN,no_ingreso,2023_2
1,"abad neyra, andrea tais",medicina humana,299.00,NaN,no_ingreso,2023_2
2,"abad oncoy, maria belen",medicina humana,441.75,NaN,no_ingreso,2023_2


-- Finalizado proceso 2023_2 (24553 filas limpias)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
-- PROCESO: 2024_1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Usando URL base: https://admision.unmsm.edu.pe/Website20241/A.html
Carreras guardadas: 83 → /content/drive/MyDrive/proyecto_admision_unmsm/data/2024_1/carreras_2024_1.csv
Resultados RAW guardados en: /content/drive/MyDrive/proyecto_admision_unmsm/data/2024_1/resultados_raw_2024_1.csv


...

-- Finalizado proceso 2025_2 (17794 filas limpias)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
-- PROCESO: 2025_2M
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Usando URL base: https://admision.unmsm.edu.pe/Website20252GeneralA/A.html
Carreras guardadas: 18 → /content/drive/MyDrive/proyecto_admision_unmsm/data/2025_2M/carreras_2025_2M.csv
Resultados RAW guardados en: /content/drive/MyDrive/proyecto_admision_unmsm/data/2025_2M/resultados_raw_2025_2M.csv


,CODIGO,APELLIDOS Y NOMBRES,ESCUELA PROFESIONAL,PUNTAJE,MERITO E.P,OBSERVACIÓN
0,877220,"ABAD CONCHA, MIKHAIL MARED MARZUK",MEDICINA HUMANA,1098.125,NaN,NaN
1,861521,"ABAD CONDORI, DANITZA ROCIO",MEDICINA HUMANA,1127.875,NaN,NaN
2,866556,"ABAD GOMEZ, EVELYN DAYANNA",MEDICINA HUMANA,818.875,NaN,NaN


Resultados LIMPIOS guardados en: /content/drive/MyDrive/proyecto_admision_unmsm/data/2025_2M/resultados_limpio_2025_2M.csv


,nombre,carrera_1,puntaje,merito,obs,proceso
0,"abad concha, mikhail mared marzuk",medicina humana,1098.125,NaN,no_ingreso,2025_2
1,"abad condori, danitza rocio",medicina humana,1127.875,NaN,no_ingreso,2025_2
2,"abad gomez, evelyn dayanna",medicina humana,818.875,NaN,no_ingreso,2025_2


-- Finalizado proceso 2025_2M (9836 filas limpias)



In [11]:
base_path = Path("/content/drive/MyDrive/proyecto_admision_unmsm/data")

# Unir los dos datasets del 2025_2
df_2025_2 = pd.read_csv(base_path / "2025_2" / "resultados_limpio_2025_2.csv")
df_2025_2M = pd.read_csv(base_path / "2025_2M" / "resultados_limpio_2025_2M.csv")

# Combinar ambos (tienen mismas columnas)
df_2025_2 = pd.concat([df_2025_2, df_2025_2M], ignore_index=True)
df_2025_2.to_csv(base_path / "2025_2" / "resultados_limpio_2025_2.csv", index=False)
print(f"Fusionadas 2025_2 + 2025_2M → {df_2025_2.shape[0]} filas totales")

Fusionadas 2025_2 + 2025_2M → 27630 filas totales


# 8. CREACION DE DF_MASTER

In [12]:
base_path = Path("/content/drive/MyDrive/proyecto_admision_unmsm/data")

# Ya no usamos 2025_2M porque ya lo unimos al 2025_2
procesos = ["2023_2", "2024_1", "2024_2","2025_1","2025_2"]

dfs = []
for proceso in procesos:
    file_path = base_path / proceso / f"resultados_limpio_{proceso}.csv"
    df = pd.read_csv(file_path)
    dfs.append(df)
    print(f"Cargado {proceso}: {df.shape[0]} filas")

# Combinar en uno solo
df_master = pd.concat(dfs, ignore_index=True)
print(f"\n  DataFrame maestro creado: {df_master.shape[0]} filas totales")
print(f"Cantidad de carreras unicas antes de normalizarlas: {len(df_master['carrera_1'].unique())}")


Cargado 2023_2: 24553 filas
Cargado 2024_1: 28862 filas
Cargado 2024_2: 22077 filas
Cargado 2025_1: 28665 filas
Cargado 2025_2: 27630 filas

  DataFrame maestro creado: 131787 filas totales
Cantidad de carreras unicas antes de normalizarlas: 128


 Nos topamos con un error, el error es que hay carreras que se repiten porque para una misma carrera hay dos nombres en dos procesos distintos

In [13]:
for i, carrera in enumerate(sorted(df_master["carrera_1"].dropna().unique()), start=1):
    print(f"{i}. {carrera}")

1. administracion - chilca
2. administracion - huaral
3. administracion - lima
4. administracion - s.j.l
5. administracion - villa rica
6. administracion de la gastronomia
7. administracion de negocios internacionales - huaral
8. administracion de negocios internacionales - lima
9. administracion de negocios internacionales - s.j.l
10. administracion de turismo - huaral
11. administracion de turismo - lima
12. administracion de turismo - s.j.l
13. administracion maritima y portuaria
14. administracion maritima y portuaria - chancay
15. antropologia
16. arqueologia
17. arquitectura y urbanismo
18. arte
19. auditoria empresarial y del sector publico
20. auditoria empresarial y del sector publico - lima
21. auditoria empresarial y del sector publico - s.j.l
22. bibliotecologia y ciencias de la informacion
23. ciencia de la computacion
24. ciencia politica
25. ciencias biologicas
26. ciencias biologicas - lima
27. ciencias de la computacion
28. ciencias de los alimentos
29. computacion cie

In [14]:
len(df_master['carrera_1'].unique())

128

In [15]:
reemplazos_carreras = {
    "auditoria empresarial y del sector publico - lima": "auditoria empresarial y del sector publico",
    "ciencias biologicas - lima": "ciencias biologicas",
    "ciencia de la computacion": "ciencias de la computacion",
    "derecho - lima": "derecho",
    "educacion fisica - lima": "educacion fisica",
    "gestion tributaria - lima": "gestion tributaria",
    "ingenieria agroindustrial - lima": "ingenieria agroindustrial",
    "contabilidad - lima": "contabilidad",
    "ingenieria de minas - lima": "ingenieria de minas",
    "ingenieria electrica - lima": "ingenieria electrica",
    "ingenieria geologica - lima": "ingenieria geologica",
    "nutricion - lima": "nutricion",
    "presupuesto y finanzas publicas - lima": "presupuesto y finanzas publicas",
    "psicologia - lima": "psicologia"
}

df_master["carrera_1"] = df_master["carrera_1"].replace(reemplazos_carreras)

if "carrera_2" in df_master.columns:
    df_master["carrera_2"] = df_master["carrera_2"].replace(reemplazos_carreras)
    df_master["carrera_2"] = df_master["carrera_2"].astype("string")

print("-- Carreras normalizadas correctamente.")


-- Carreras normalizadas correctamente.


In [16]:
for i, carrera in enumerate(sorted(df_master["carrera_1"].dropna().unique()), start=1):
    print(f"{i}. {carrera}")

1. administracion - chilca
2. administracion - huaral
3. administracion - lima
4. administracion - s.j.l
5. administracion - villa rica
6. administracion de la gastronomia
7. administracion de negocios internacionales - huaral
8. administracion de negocios internacionales - lima
9. administracion de negocios internacionales - s.j.l
10. administracion de turismo - huaral
11. administracion de turismo - lima
12. administracion de turismo - s.j.l
13. administracion maritima y portuaria
14. administracion maritima y portuaria - chancay
15. antropologia
16. arqueologia
17. arquitectura y urbanismo
18. arte
19. auditoria empresarial y del sector publico
20. auditoria empresarial y del sector publico - s.j.l
21. bibliotecologia y ciencias de la informacion
22. ciencia politica
23. ciencias biologicas
24. ciencias de la computacion
25. ciencias de los alimentos
26. computacion cientifica
27. computacion cientifica - chilca
28. comunicacion social
29. conservacion y restauracion
30. contabilida

In [17]:
df_master.columns


Index(['nombre', 'carrera_1', 'puntaje', 'merito', 'obs', 'proceso',
       'carrera_2'],
      dtype='object')

In [18]:
# Separamos la columna 'nombres' en 'apellidos' y 'nombres'
df_master[['apellidos', 'nombres']] = df_master['nombre'].str.split(',', n=1, expand=True)
df_master['apellidos'] = df_master['apellidos'].str.strip()
df_master['nombres'] = df_master['nombres'].str.strip()
df_master.drop(columns=['nombre'], inplace=True)


In [19]:
# Ordenamos columnas
df_master = df_master[['apellidos', 'nombres', 'carrera_1', 'carrera_2', 'puntaje', 'merito', 'obs', 'proceso']]
df_master.head()


,apellidos,nombres,carrera_1,carrera_2,puntaje,merito,obs,proceso
0,abad granda,andre fahet,medicina humana,<NA>,774.750,NaN,no_ingreso,2023_2
1,abad neyra,andrea tais,medicina humana,<NA>,299.000,NaN,no_ingreso,2023_2
2,abad oncoy,maria belen,medicina humana,<NA>,441.750,NaN,no_ingreso,2023_2
3,abal principe,ingrid,medicina humana,<NA>,530.500,NaN,no_ingreso,2023_2
4,abanto araujo,percy angel,medicina humana,<NA>,584.625,NaN,no_ingreso,2023_2


In [20]:
df_master.to_csv(base_path / "resultados_limpio_total.csv", index=False, encoding="utf-8-sig")
print(f" Guardado DataFrame maestro en: {base_path / 'resultados_limpio_total.csv'}")

 Guardado DataFrame maestro en: /content/drive/MyDrive/proyecto_admision_unmsm/data/resultados_limpio_total.csv


# Añadimos proceso 2026-I

In [21]:
base_path = Path("/content/drive/MyDrive/proyecto_admision_unmsm/data")
df = pd.read_csv(base_path / "resultados_limpio_total.csv")
df.head()

/tmp/ipykernel_6161/2777132220.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(base_path / "resultados_limpio_total.csv")


,apellidos,nombres,carrera_1,carrera_2,puntaje,merito,obs,proceso
0,abad granda,andre fahet,medicina humana,NaN,774.750,NaN,no_ingreso,2023_2
1,abad neyra,andrea tais,medicina humana,NaN,299.000,NaN,no_ingreso,2023_2
2,abad oncoy,maria belen,medicina humana,NaN,441.750,NaN,no_ingreso,2023_2
3,abal principe,ingrid,medicina humana,NaN,530.500,NaN,no_ingreso,2023_2
4,abanto araujo,percy angel,medicina humana,NaN,584.625,NaN,no_ingreso,2023_2


In [22]:
df.columns

Index(['apellidos', 'nombres', 'carrera_1', 'carrera_2', 'puntaje', 'merito',
       'obs', 'proceso'],
      dtype='object')

In [23]:
df_26_1 = pd.read_csv(base_path/"2026_1/resultados_raw_2026_1.csv")
df_26_1.head()

,web_scraper_order,web_scraper_start_url,link_carreras,paginacion,codigo,Nombre,Escuela,Puntaje,merito_ep,Observación
0,1772682336-1,https://admision.unmsm.edu.pe/Website20261/A/A...,https://admision.unmsm.edu.pe/Website20261/A/1...,NaN,830038,"ALEJOS PISCO, ABIGAIL MARIACLARA",TRABAJO SOCIAL,830.500,NaN,NaN
1,1772682336-2,https://admision.unmsm.edu.pe/Website20261/A/A...,https://admision.unmsm.edu.pe/Website20261/A/1...,NaN,802187,"ALVARADO TAPIA, MARYCIELO YAMELY",TRABAJO SOCIAL,669.250,NaN,NaN
2,1772682336-3,https://admision.unmsm.edu.pe/Website20261/A/A...,https://admision.unmsm.edu.pe/Website20261/A/1...,NaN,799314,"ANDRADE LA MADRID, ARIANA SAMAR",TRABAJO SOCIAL,612.000,NaN,NaN
3,1772682336-4,https://admision.unmsm.edu.pe/Website20261/A/A...,https://admision.unmsm.edu.pe/Website20261/A/1...,NaN,709194,"AQUINO SILVA, CARLOS JUNIOR",TRABAJO SOCIAL,1021.375,7.0,ALCANZÓ VACANTE
4,1772682336-5,https://admision.unmsm.edu.pe/Website20261/A/A...,https://admision.unmsm.edu.pe/Website20261/A/1...,NaN,696655,"ASCATE MELENDREZ, ROMINA",TRABAJO SOCIAL,970.000,23.0,ALCANZÓ VACANTE


In [24]:
df_26_1.columns

Index(['web_scraper_order', 'web_scraper_start_url', 'link_carreras',
       'paginacion', 'codigo', 'Nombre', 'Escuela', 'Puntaje', 'merito_ep',
       'Observación'],
      dtype='object')

In [25]:
# Hacer que las columnas sean iguales
#'apellidos', 'nombres', 'carrera_1', 'carrera_2', 'puntaje', 'merito', 'obs', 'proceso'
df_26_1 = df_26_1.drop(columns=['web_scraper_order','web_scraper_start_url', 'link_carreras',
       'paginacion', 'codigo'])
df_26_1.head()

,Nombre,Escuela,Puntaje,merito_ep,Observación
0,"ALEJOS PISCO, ABIGAIL MARIACLARA",TRABAJO SOCIAL,830.500,NaN,NaN
1,"ALVARADO TAPIA, MARYCIELO YAMELY",TRABAJO SOCIAL,669.250,NaN,NaN
2,"ANDRADE LA MADRID, ARIANA SAMAR",TRABAJO SOCIAL,612.000,NaN,NaN
3,"AQUINO SILVA, CARLOS JUNIOR",TRABAJO SOCIAL,1021.375,7.0,ALCANZÓ VACANTE
4,"ASCATE MELENDREZ, ROMINA",TRABAJO SOCIAL,970.000,23.0,ALCANZÓ VACANTE


In [26]:
# Separamos la columna 'nombres' en 'apellidos' y 'nombres'
df_26_1[['apellidos', 'nombres']] = df_26_1['Nombre'].str.split(',', n=1, expand=True)
df_26_1['apellidos'] = df_26_1['apellidos'].str.strip()
df_26_1['nombres'] = df_26_1['nombres'].str.strip()
df_26_1.drop(columns=['Nombre'], inplace=True)

In [27]:
df_26_1.columns

Index(['Escuela', 'Puntaje', 'merito_ep', 'Observación', 'apellidos',
       'nombres'],
      dtype='object')

In [28]:
df_26_1 = df_26_1.rename(columns={
    'merito_ep': 'merito',
    'Observación': 'obs',
    'Escuela': 'carrera_1',
    'Puntaje': 'puntaje'
    })

In [29]:
df_26_1['carrera_2'] = np.nan
df_26_1['proceso'] = '2026_1'

In [30]:
# Ordenamos columnas
#'apellidos', 'nombres', 'carrera_1', 'carrera_2', 'puntaje', 'merito', 'obs', 'proceso'
df_26_1 = df_26_1[['apellidos', 'nombres', 'carrera_1', 'carrera_2', 'puntaje', 'merito', 'obs', 'proceso']]
df_26_1.head()

,apellidos,nombres,carrera_1,carrera_2,puntaje,merito,obs,proceso
0,ALEJOS PISCO,ABIGAIL MARIACLARA,TRABAJO SOCIAL,NaN,830.500,NaN,NaN,2026_1
1,ALVARADO TAPIA,MARYCIELO YAMELY,TRABAJO SOCIAL,NaN,669.250,NaN,NaN,2026_1
2,ANDRADE LA MADRID,ARIANA SAMAR,TRABAJO SOCIAL,NaN,612.000,NaN,NaN,2026_1
3,AQUINO SILVA,CARLOS JUNIOR,TRABAJO SOCIAL,NaN,1021.375,7.0,ALCANZÓ VACANTE,2026_1
4,ASCATE MELENDREZ,ROMINA,TRABAJO SOCIAL,NaN,970.000,23.0,ALCANZÓ VACANTE,2026_1


In [31]:
# Limpieza general de texto
for col in df_26_1.select_dtypes(include="object").columns:
    df_26_1[col] = (
        df_26_1[col].astype(str)
        .str.strip()
        .str.lower()
        .apply(unidecode.unidecode)
    )

if "obs" in df_26_1.columns:
    df_26_1["obs"] = df_26_1["obs"].replace({
        'alcanzo vacante': 'ingreso',
        'alcanzo vacante primera opcion': 'ingreso',
        'alcanzo vacante segunda opcion': 'ingreso_2',
        'nan': 'no_ingreso'
    })

In [32]:
df_26_1.describe()

,carrera_2,puntaje,merito
count,0.0,26179.000000,2772.000000
mean,NaN,805.517561,22.882395
std,NaN,202.755801,21.905539
min,NaN,144.500000,1.000000
25%,NaN,661.125000,8.000000
50%,NaN,790.125000,16.000000
75%,NaN,933.750000,31.000000
max,NaN,1679.125000,136.000000


In [33]:
df_master.shape

(131787, 8)

In [34]:
df_26_1.shape

(26518, 8)

In [35]:
df_total = pd.concat([df_master, df_26_1])
df_total.shape

/tmp/ipykernel_6161/858112843.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total = pd.concat([df_master, df_26_1])


(158305, 8)

In [36]:
df_total.to_csv(base_path / "resultados_limpio_total_2.csv", index=False, encoding="utf-8-sig")
print(f" Guardado DataFrame maestro en: {base_path / 'resultados_limpio_total_2.csv'}")

 Guardado DataFrame maestro en: /content/drive/MyDrive/proyecto_admision_unmsm/data/resultados_limpio_total_2.csv


In [37]:
df_total.head()
df_total.info()
df_total.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 158305 entries, 0 to 26517
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   apellidos  158305 non-null  object 
 1   nombres    158305 non-null  object 
 2   carrera_1  158305 non-null  object 
 3   carrera_2  815 non-null     string 
 4   puntaje    156436 non-null  float64
 5   merito     17665 non-null   float64
 6   obs        158305 non-null  object 
 7   proceso    158305 non-null  object 
dtypes: float64(2), object(5), string(1)
memory usage: 10.9+ MB


,puntaje,merito
count,156436.000000,17665.000000
mean,769.773930,27.100368
std,236.731936,26.138457
min,0.000000,1.000000
25%,601.500000,9.000000
50%,761.000000,19.000000
75%,930.000000,37.000000
max,1717.370000,204.000000
